In [ ]:
from astropy.table import Table
from astropy.io import fits
import astropy.coordinates as coord
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic_2d
import gala.potential as gp
import gala.dynamics as gd
from matplotlib.patches import Polygon

## Finding and reading in files

The AuriDESI files are located on NERSC at `/global/cfs/cdirs/desi/users/namitha/auridesi/v1`. The exact mock you want is identified by:
* halo number: there are 6 Auriga haloes (numbered 6, 16, 21, 23, 24, 27) that were mock observed to make AuriDESI
* angle: each halo was observed from 4 perspectives located within the disk, at different azimuthal angles (30, 120, 210, 300) relative to the z=0 bar in the simulation
* observing strategy: mocks generated from running fiberassign are in `mock_spectroscopic_catalogs`, while mocks that assign fibers randomly preserving the ratio of BLUE/RED/BROAD stars are in `mock_randomly_sampled_catalogs`

Let's start simple with halo 21 and looking at the header

In [ ]:
fname = '/global/cfs/cdirs/desi/users/namitha/auridesi/v1/mock_spectroscopic_catalogs/030_deg/H21_030deg_mock.fits'
header = fits.open(fname)[0].header
header


All of this is pretty straightforward, and we see that the coordinate system defining solar radius, motion, etc are also here. Many of these quantities (like UVW motion) are consistent across all angles and even different haloes, but to be safe I recommend always accessing them via this header.

The other HDUs here are the actual mock data. The first three HDUs have columns that match what they mean in the real DESI data releases, while the final two have columns that are relevant to the simulation. In index order:
* `RVTAB`: radial velocity, log g, Teff, alpha
* `FIBERMAP`: RA, Dec, proper motion, photometry (both grz and Gaia)
* `GAIA`: RA, Dec, parallax, proper motion
* `TRUE_VALUE`: for each mock star, the underlying properties (e.g. radial velocity, Teff, proper motion) _before_ convolving with observational uncertainties, as well as simulation metadata like the ID of the parent star particle and ID of the progenitor galaxy the star particle formed inside
* `PROGENITORS`: for each progenitor galaxy, properties like mass (DM, stellar, peak, bound at z=0), accretion time

More detailed explanations of each column are in the `Data model.txt` file also in the directory.

In [ ]:
# just load everything real quick
rvtab = Table.read(fname, hdu='rvtab')
fibermap = Table.read(fname, hdu='fibermap')
gaia = Table.read(fname, hdu='gaia')
true = Table.read(fname, hdu='true_value')
prog = Table.read(fname, hdu='progenitors')


For a quick illustration of the difference between the fiberassign mock and random mock

In [ ]:
# random mock fibermap
fname_r = '/global/cfs/cdirs/desi/users/namitha/auridesi/v1/mock_randomly_sampled_catalogs/030_deg/H21_030deg_mock.fits'
fibermap_r = Table.read(fname_r, hdu='fibermap')

# MWS numbers from Table 2 of Cooper+2023
N_red = 805794
N_blue = 3693518
N_broad = 2077222
total = N_red + N_blue + N_broad
print(f"Cooper+2023: {total} stars, {(N_blue / total):.3f} blue / {(N_red / total):.3f} red / {(N_broad / total):.3f} broad")

# mock numbers
for fmap, label in zip([fibermap, fibermap_r], ['fiberassign', 'rand-assign']):
    f_broad = np.sum((fmap['MWS_TARGET'] & (1 << 0)) > 0) / len(fmap)
    f_blue = np.sum((fmap['MWS_TARGET'] & (1 << 8)) > 0) / len(fmap)
    f_red = np.sum((fmap['MWS_TARGET'] & (1 << 11)) > 0) / len(fmap)
    print(f'{label}: {len(fmap)} stars, {f_blue:.3f} blue / {f_red:.3f} red / {f_broad:.3f} broad')


## Identifying tracer populations

Generally, if a tracer sample (BHB, RRL, RGB) can be identified using stellar properties like log g, Teff, then it's straightforward to apply the same cuts to the mock data.

I'll show an example for BHBs, along with some common pitfalls. We'll start using the "true" values.

In [ ]:
# get the distance (this works because we are using the true, not error-convolved, parallax)
dist = coord.Distance(parallax=true['PARALLAX'])
true['GMAG'] = true['APP_GMAG'] - dist.distmod.value


In [ ]:
# plot the selection, shade in Amanda's BHB cuts
fig, axes = plt.subplots(1, 2)
axes[0].hist2d(true['TEFF'], true['LOGG'], bins=100, norm='log', range=[[2000,12000], [0, 5.5]])
axes[0].fill_between(x=[12000, 7000], y1=3.5, y2=0, alpha=0.2)
axes[0].invert_yaxis()
axes[0].invert_xaxis()
axes[0].set_xlabel('True Teff [K]')
axes[0].set_ylabel('True logg')

axes[1].hist2d(true['TEFF'], true['GMAG'], bins=100, norm='log', range=[[2000,12000], [-3, 17]])
axes[1].invert_yaxis()
axes[1].set_xlabel('True Teff [K]')
axes[1].set_ylabel('True Mg [mag]')

fig.tight_layout();


Here we see that in addition to the familiar MSTO and horizontal branch, we have a bunch of other stars at high logg and Teff. This is recent star formation activity in a satellite galaxy that happens to fall inside the DESI footprint. If you reload the notebook with Halo 6 instead, most of this goes away and the CMD is much cleaner.

Looking by eye, it seems we can draw some boxes that will isolate the BHB sample, mostly by hugging the horizontal branch in log g and Teff. After some iteration, we land on this:

In [ ]:
box1 = [[9750,3.7], [9750,3.3], [5500,2.2], [5500,2.7]]
p1 = Polygon(box1, facecolor='None', edgecolor='r')
# box2 = [[9750,0.3], [9750,1], [5000,1], [5000,0.3]]
box2 = [[9750,0.45], [9750,0.65], [5000,0.65], [5000,0.45]]
p2 = Polygon(box2, facecolor='None', edgecolor='r')

in_box1 = p1.get_path().contains_points(true[['TEFF', 'LOGG']].to_pandas().to_numpy())
in_box2 = p2.get_path().contains_points(true[['TEFF', 'GMAG']].to_pandas().to_numpy())
print(np.sum(in_box1 & in_box2))

# plot the selection
fig, axes = plt.subplots(1, 2)
# axes[0].hist2d(true['TEFF'], true['LOGG'], bins=100, norm='log', range=[[2000,12000], [0, 5.5]])
axes[0].hist2d(true['TEFF'][in_box2], true['LOGG'][in_box2], bins=100, norm='log', range=[[2000,12000], [0, 5.5]])
axes[0].add_patch(p1)
axes[0].fill_between(x=[12000, 7000], y1=3.5, y2=0, alpha=0.2)
axes[0].invert_yaxis()
axes[0].invert_xaxis()
axes[0].set_xlabel('True Teff [K]')
axes[0].set_ylabel('True logg')

# axes[1].hist2d(true['TEFF'], true['GMAG'], bins=100, norm='log', range=[[2000,12000], [-3, 17]])
axes[1].hist2d(true['TEFF'][in_box1], true['GMAG'][in_box1], bins=100, norm='log', range=[[2000,12000], [-3, 17]])
axes[1].invert_yaxis()
axes[1].add_patch(p2)
axes[1].set_xlabel('True Teff [K]')
axes[1].set_ylabel('True Mg [mag]')

fig.tight_layout();


In the plot above, in each panel I'm showing the stars that remain after you apply the Polygon selection in the other panel. It's only with a combination of cuts that you get something that looks like it's tracing a population that we could call "BHBs." If you're uncomfortable with BHBs taking up the entire horizontal branch, you could apply an addition Teff cut to exclude stars cooler than some threshold.

We could look at similar validation plots in things like age and color-color

In [ ]:
g_r = true['APP_GMAG'] - true['APP_RMAG']
r_z = true['APP_RMAG'] - true['APP_ZMAG']

fig, axes = plt.subplots(1, 3)
sels = [in_box1, in_box2, (in_box1 & in_box2)]
labels = ['logg-Teff', 'absmag', 'both']
for sel, label, ax in zip(sels, labels, axes):
    ax.hist2d(g_r[sel], r_z[sel], range=[[-0.5,1.5], [-0.5, 1.25]], bins=100, norm='log')
    ax.set_xlabel('g - r')
    ax.set_ylabel('r - z')
    ax.set_title(label)

fig.tight_layout();

In [ ]:
agebins = np.linspace(0, 14, 100)

for sel in [in_box1, in_box2, (in_box1 & in_box2)]:
    plt.hist(true['AGE'][sel], bins=agebins, histtype='step')
plt.xlabel('Age [Gyr]');

If the goal is to test how just using BHBs as a tracer population works for a given method, this selection based on true quantities might be enough! If the idea is to get a sense of what the effects of observational contamination, making the same cuts based on observed properties (maybe replacing the absolute magnitude cut above with a cut in color-color space, then cheating a little and verify by looking at the resulting distribution in true absolute magnitude or age) would be a good way forward. Again, depends on the science question!

## 6D phase space

Here's a few examples of using the 6D phase space measurements provided in the catalog.

In [ ]:
# coord system from header
galcen_v_sun = [header['U_SUN'], header['VLSR']+header['V_SUN'], header['W_SUN']]*u.km/u.s
zsun = header['SOLAR_H']*u.Mpc
galcen_dist = -header['SOLAR_R']*u.Mpc
galcen_frame = coord.Galactocentric(galcen_v_sun=galcen_v_sun, z_sun=zsun,
                                    galcen_distance=galcen_dist)

# cheat and use true distances rather than invert observed parallax
dist = coord.Distance(parallax=true['PARALLAX']).to(u.kpc)

sc = coord.SkyCoord(ra=gaia['RA'], dec=gaia['DEC'], distance=dist,
                    pm_ra_cosdec=gaia['PMRA'],
                    pm_dec=gaia['PMDEC'],
                    radial_velocity=rvtab['VRAD'], frame=coord.ICRS)
galcen = sc.transform_to(galcen_frame)

Note that we cheat here and use the true distances. If we wanted to add uncertainties that we think are understood (e.g. "BHBs have 10% uncertainties"), we could add lines that do something like
```
dist = coord.Distance(parallax=true['PARALLAX']*u.mas).to(u.kpc)
rng = np.random.default_rng()
dist *= rng.normal(loc=1, scale=0.1, size=len(dist))
```
For more complicated cases (e.g. Bailer-Jones-like), it's not so clear what to do. Alex might have a few ideas if people want something like this

In [ ]:
energy = 0.5*galcen.velocity.norm()**2 + true['GRAV_POTENTIAL']
Lz = (galcen.x * galcen.v_y) - (galcen.y * galcen.v_x)

In [ ]:
prog.sort('MSTAR_TOT')
prog = prog[::-1]

fig, axes = plt.subplots(1, 5, sharex=True, sharey=True, figsize=(12, 6))
for prog_id, ax in zip(prog['TREE_ID'][:5], axes):
    sel = true['TREE_ID'] == prog_id
    ax.hist2d(Lz[sel].to(u.kpc**2/u.Myr).value, energy[sel].to(u.kpc**2/u.Myr**2).value, bins=300, vmin=1, range=[[-5, 5], [-0.2, 0]], norm='log')

## Orbits

One annoyance is that the mock catalogs generate stars at z=0, and they do not provide dynamical quantities like apo/pericenter or eccentricity. We can address this by integrating orbits in a representation of the potential based on the simulation, which can be done in increasing complexity (eg symmetric static potentials, static basis-function expansions, time-dependent expansions).

To start off (and to reasonably match common practices so far in Milky Way observations), I've generated three-component potentials (bulge, disk, halo) that have been fit to the cumulative mass distribution of the z=0 simulation snapshot. They are located in `/global/cfs/cdirs/desi/users/ahriley/auriga/composite-potential-fits/`. There's two versions of this - ones that place no constraints on the physical parameters of the components, and another that prevents things like the disk component shrinking to ~1 kpc to replace the bulge. For more details, see `fit-potential.py` in the same directory.

In [ ]:
def get_potential(halo, bounded=True):
    folder = '/global/cfs/cdirs/desi/users/ahriley/auriga/composite-potential-fits/'
    boundstr = '-bounded' if bounded else ''
    fname = folder + f'halo{halo}-level3{boundstr}.yaml'

    pot = gp.load(fname)
    return pot

# load potential as a gala representation
pot = get_potential(halo=6)

# can convert that to galpy if you prefer
# pot_galpy = pot.to_galpy_potential()

# haven't tried this myself, but apparently this also works for agama
# using gala to do the conversion may also work soon, see: https://github.com/adrn/gala/pull/341
# pot_agama = agama.GalaPotential(gpot1)

Now that we have a representation of the potential (even if it isn't _THE_ potential), we can integrate orbits as normal, using either observed or true quantities

In [ ]:
# calculate orbits for a subset of the mocks, just for speed
rng = np.random.default_rng(seed=42)
sel = rng.choice(len(sc), size=10000)
w0 = gd.PhaseSpacePosition(galcen[sel].data)


In [ ]:
# integrate orbits for 1 Gyr (for 10^4 stars takes about 70 seconds)
orbits = pot.integrate_orbit(w0, dt=-1*u.Myr, n_steps=1000)
apo = orbits.apocenter(approximate=False)
ecc = orbits.eccentricity(approximate=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

kwargs = {'cmap': 'Spectral_r', 'vmin': -1, 'vmax': 0.25, 'rasterized': True}

zvals = rvtab['FEH'][sel]

xe = np.linspace(0,1,50)
ye = np.logspace(np.log10(5),np.log10(100),50)
X,Y = np.meshgrid(xe[:-1],ye[:-1])
xvals = ecc.value
yvals = apo.to(u.kpc).value
hist = binned_statistic_2d(x=xvals, y=yvals, values=zvals, bins=[xe, ye], statistic='count')[0]
im1 = axes[0].pcolormesh(X, Y, np.log10(hist.T))
axes[0].set_xlabel(r'Eccentricity')
axes[0].set_ylabel(r'R$_{apo}$ [kpc]')

hist = binned_statistic_2d(x=xvals, y=yvals, values=zvals, bins=[xe, ye], statistic='median')[0]
im2 = axes[1].pcolormesh(X, Y, hist.T, **kwargs)
axes[1].set_xlabel(r'Eccentricity')
axes[1].set_ylabel(r'R$_{apo}$ [kpc]')

for ax in axes:
    ax.set_yscale('log')
    ax.set_ylim(5,100)

fig.colorbar(im2, label=r'Median [Fe/H]')

plt.tight_layout()


While it's poorly sampled since we only used 10^4 stars, we can already start to see chemodynamical trends that match what's seen in the Milky Way

### Pulling out satellites

It can sometimes be useful to identify accretion events or other structures, either for removing or for diagnostics. There are two flags that are useful here

* `POP_ID` is 0 for stars that form in the main host, 1 if a star was accreted as is no longer bound to a satellite, 2 if it is currently bound to a satellite
* `TREE_ID` is an integer label for each accretion event (0 is the main host)
* `SUBHALO_NR` is an integer label for the present-day bound structures (0 is the main host)

In [ ]:
# bound to satellite at z=0
sel = true['POP_ID'] == 2

In [ ]:
plt.scatter(rvtab['TARGET_RA'][sel], rvtab['TARGET_DEC'][sel], s=1)
plt.xlim(0, 360)
plt.ylim(-20, 90)
plt.xlabel('RA [deg]')
plt.ylabel('Dec [deg]');